In [218]:
import os, glob, json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter, defaultdict
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.model_selection import train_test_split

import random
#https://github.com/ohoaha/can-ids-benchmarking-road/blob/main/ML_Anomaly_experiments.ipynb 

In [219]:
AMBIENT_DATA = [
"road/signal_extractions/ambient/accelerator_attack_drive_1.csv",
"road/signal_extractions/ambient/accelerator_attack_drive_2.csv",
"road/signal_extractions/ambient/accelerator_attack_reverse_1.csv",
"road/signal_extractions/ambient/accelerator_attack_reverse_2.csv",
"road/signal_extractions/ambient/ambient_dyno_drive_basic_long.csv",
"road/signal_extractions/ambient/ambient_dyno_drive_basic_short.csv",
"road/signal_extractions/ambient/ambient_dyno_drive_benign_anomaly.csv",
"road/signal_extractions/ambient/ambient_dyno_drive_extended_long.csv",
"road/signal_extractions/ambient/ambient_dyno_drive_extended_short.csv",
"road/signal_extractions/ambient/ambient_dyno_drive_radio_infotainment.csv",
"road/signal_extractions/ambient/ambient_dyno_drive_winter.csv",
"road/signal_extractions/ambient/ambient_dyno_exercise_all_bits.csv",
"road/signal_extractions/ambient/ambient_dyno_idle_radio_infotainment.csv",
"road/signal_extractions/ambient/ambient_dyno_reverse.csv",
"road/signal_extractions/ambient/ambient_highway_street_driving_diagnostics.csv",
"road/signal_extractions/ambient/ambient_highway_street_driving_long.csv",
"road/signal_extractions/ambient/correlated_signal_attack_1_masquerade.csv",
"road/signal_extractions/ambient/correlated_signal_attack_2_masquerade.csv",
"road/signal_extractions/ambient/correlated_signal_attack_3_masquerade.csv",
"road/signal_extractions/ambient/max_engine_coolant_temp_attack_masquerade.csv",
"road/signal_extractions/ambient/max_speedometer_attack_1_masquerade.csv",
"road/signal_extractions/ambient/max_speedometer_attack_2_masquerade.csv",
"road/signal_extractions/ambient/max_speedometer_attack_3_masquerade.csv",
"road/signal_extractions/ambient/reverse_light_off_attack_1_masquerade.csv",
"road/signal_extractions/ambient/reverse_light_off_attack_2_masquerade.csv",
"road/signal_extractions/ambient/reverse_light_off_attack_3_masquerade.csv",
"road/signal_extractions/ambient/reverse_light_on_attack_1_masquerade.csv",
"road/signal_extractions/ambient/reverse_light_on_attack_2_masquerade.csv",
"road/signal_extractions/ambient/reverse_light_on_attack_3_masquerade.csv",
] 

ATTACK_DATA = [
"road/signal_extractions/attacks/accelerator_attack_drive_1.csv",
"road/signal_extractions/attacks/accelerator_attack_drive_2.csv",
"road/signal_extractions/attacks/accelerator_attack_reverse_1.csv",
"road/signal_extractions/attacks/accelerator_attack_reverse_2.csv",
"road/signal_extractions/attacks/correlated_signal_attack_1_masquerade.csv",
"road/signal_extractions/attacks/correlated_signal_attack_2_masquerade.csv",
"road/signal_extractions/attacks/correlated_signal_attack_3_masquerade.csv",
"road/signal_extractions/attacks/max_engine_coolant_temp_attack_masquerade.csv",
"road/signal_extractions/attacks/max_speedometer_attack_1_masquerade.csv",
"road/signal_extractions/attacks/max_speedometer_attack_2_masquerade.csv",
"road/signal_extractions/attacks/max_speedometer_attack_3_masquerade.csv",
"road/signal_extractions/attacks/reverse_light_off_attack_1_masquerade.csv",
"road/signal_extractions/attacks/reverse_light_off_attack_2_masquerade.csv",
"road/signal_extractions/attacks/reverse_light_off_attack_3_masquerade.csv",
"road/signal_extractions/attacks/reverse_light_on_attack_1_masquerade.csv",
"road/signal_extractions/attacks/reverse_light_on_attack_2_masquerade.csv",
"road/signal_extractions/attacks/reverse_light_on_attack_3_masquerade.csv",
]
print(len(ATTACK_DATA), len(AMBIENT_DATA))
# combined list used by the rest of the pipeline
DATA_FILES = AMBIENT_DATA + ATTACK_DATA
#https://github.com/ohoaha/can-ids-benchmarking-road/blob/main/ML_Anomaly_experiments.ipynb

17 29


In [220]:
def load_capture(path, test=False):
    if test:
        path = 'road/' + path
    print("Reading in: ", path)
    df = pd.read_csv(path)
    # enforce canonical columns
    df = df.rename(columns=lambda c: c.strip())
    # ensure Time is float seconds
    df["Time"] = df["Time"].astype(float)
    # sort by time (important for resampling)
    df = df.sort_values("Time").reset_index(drop=True)
    # forward-fill signal NaNs
    signal_cols = [c for c in df.columns if c.startswith("Signal_")]
    df[signal_cols] = df[signal_cols].ffill().fillna(0) #remove all NAs
    return df

df_test = load_capture(ATTACK_DATA[1], test = True)
print(df_test['Label'].value_counts())

Reading in:  road/road/signal_extractions/attacks/accelerator_attack_drive_2.csv
Label
0    161371
Name: count, dtype: int64


In [221]:
def interpolate_capture(df, target_hz=200):
    t_min, t_max = df["Time"].min(), df["Time"].max()
    # construct uniform time grids
    new_time = np.arange(t_min, t_max, 1.0 / target_hz)
    # reindex to grid
    interp_df = pd.DataFrame({"Time": new_time})
    # only interpolate the signal columns
    signal_cols = [c for c in df.columns if c.startswith("Signal_")]
    for col in signal_cols:
        interp_df[col] = np.interp(new_time, df["Time"].values, df[col].astype(float).values)
    interp_df = pd.merge_asof(interp_df, df[["Time","ID"]], on="Time") #handle IDs by setting them to nearest neighbor
    label_value = int(df["Label"].iloc[0])  # assign 0 or 1 to entire dataframe, TODO may want to reconsider if the data is mixed within datasets
    interp_df["Label"] = label_value
    return interp_df
# df_test = interpolate_capture(df_test, target_hz=200) #may need to tweak after testing with models 


In [ ]:
def parse_metadata():
    pass
something = pd.read_json("road/road/signal_extractions/attacks/metadata.json") 
flipped = something.T
flipped.reset_index(inplace=True)
flipped.columns = ['type', 'description', 'elapsed_sec', 'injection_data_str', 'injection_id',
       'injection_interval', 'modified', 'on_dyno']
meta_df = flipped[['type', 'elapsed_sec', 'injection_interval', 'injection_id']]
meta_df.tail(15)

,type,elapsed_sec,injection_interval,injection_id
2,accelerator_attack_reverse_1,86.13449,None,None
3,accelerator_attack_reverse_2,105.438372,None,None
4,correlated_signal_attack_1_masquerade,33.101852,"[9.191851, 30.050109]",0x6e0
5,correlated_signal_attack_2_masquerade,28.226893,"[6.830477, 28.225908]",0x6e0
6,correlated_signal_attack_3_masquerade,16.963905,"[4.318482, 16.95706]",0x6e0
7,max_engine_coolant_temp_attack_masquerade,25.875548,"[19.979078, 24.170183]",0x4e7
8,max_speedometer_attack_1_masquerade,88.021577,"[42.009204, 66.449011]",0xd0
9,max_speedometer_attack_2_masquerade,59.696973,"[16.009225, 47.408246]",0xd0
10,max_speedometer_attack_3_masquerade,86.766667,"[9.516489, 70.587285]",0xd0
11,reverse_light_off_attack_1_masquerade,28.109869,"[16.627923, 23.347311]",0xd0


In [224]:
with open("road/road/signal_extractions/attacks/metadata.json", "r") as f: #read in the 
    metadata = json.load(f)

def create_windows(df, filename, window_sec=3, stride_sec=0.5, target_hz=200):
        # Extract signal columns
    signal_cols = [c for c in df.columns if c.startswith("Signal_")]

    window_size = int(window_sec * target_hz)
    stride = int(stride_sec * target_hz)

    # Get attack intervals from metadata
    base_name = Path(filename).name  # extract file basename
    info = metadata.get(base_name, {})
    injection_intervals = info.get("injection_interval", [])
    if injection_intervals is None:
        injection_intervals = []

    # Ensure intervals is a list of tuples
    if isinstance(injection_intervals, list) and len(injection_intervals) == 2:
        injection_intervals = [tuple(injection_intervals)]
    elif isinstance(injection_intervals, list):
        injection_intervals = [tuple(i) for i in injection_intervals]

    X, y = [], []

    times = df['Time'].values

    for start_idx in range(0, len(df) - window_size + 1, stride):
        win = df.iloc[start_idx:start_idx + window_size]
        win_start = win['Time'].iloc[0]
        win_end = win['Time'].iloc[-1]

        # Label window as attack if any part of it overlaps any injection interval
        label = 0
        for interval in injection_intervals:
            attack_start, attack_end = interval
            if win_end >= attack_start and win_start <= attack_end:
                label = 1
                break

        X.append(win[signal_cols].values)
        y.append(label)

    return np.array(X), np.array(y)
df_test = create_windows(df_test, ATTACK_DATA[0], window_sec = 5, target_hz = 200) #can play with window_sec across some models

In [225]:
def preprocess_file(path, target_hz=100, window_sec=5):
    df = load_capture(path)
    df = interpolate_capture(df, target_hz=target_hz)
    X, y = create_windows( df, path, window_sec=window_sec, target_hz=target_hz)
    return X, y

def preprocess_all_by_file(files, base_path="road/"):
    all_data = {}  # filename -> (X, y)
    for f in files:
        # print("Processing:", f)
        X, y = preprocess_file(Path(base_path) / f, target_hz = 200, window_sec = 3)
        all_data[f] = (X, y)

    return all_data
# dfs = preprocess_all_by_file(DATA_FILES)

In [226]:
def file_has_attack(xy_tuple): #dets if session has any attack labels at all
    X, y = xy_tuple
    return np.any(y == 1)

def split_files_by_session(all_data, test_ratio=0.2, seed=42):
    random.seed(seed)
    #shuffle the attack/ambient files as a whole randomly, NOT shuffling time series within each
    files = list(all_data.keys())
    
    attack_files = [f for f in files if file_has_attack(all_data[f])]
    ambient_files = [f for f in files if not file_has_attack(all_data[f])]
    random.shuffle(attack_files)
    random.shuffle(ambient_files)

    # Stratified split
    n_attack_test = max(1, int(len(attack_files) * test_ratio))
    n_ambient_test = max(1, int(len(ambient_files) * test_ratio))
    test_files = set(attack_files[:n_attack_test] + ambient_files[:n_ambient_test])
    train_files = set(attack_files[n_attack_test:] + ambient_files[n_ambient_test:])
    print("Train sessions:", len(train_files))
    print("  Attack:", sum(np.any(all_data[f][1] == 1) for f in train_files))
    print("  Benign:", sum(not np.any(all_data[f][1] == 1) for f in train_files))

    print("\nTest sessions:", len(test_files))
    print("  Attack:", sum(np.any(all_data[f][1] == 1) for f in test_files))
    print("  Benign:", sum(not np.any(all_data[f][1] == 1) for f in test_files))
    return train_files, test_files

# train_files, test_files = split_files_by_session(dfs)
#print(test_files)

In [227]:
def build_dataset_from_split(all_data, train_files, test_files):
    X_train, y_train = [], []
    X_test, y_test   = [], []
    for f, (X, y) in all_data.items():
        if f in train_files:
            X_train.append(X)
            y_train.append(y)
        else:
            X_test.append(X)
            y_test.append(y)
    X_train = np.concatenate(X_train, axis=0)
    y_train = np.concatenate(y_train, axis=0)
    X_test  = np.concatenate(X_test, axis=0)
    y_test  = np.concatenate(y_test, axis=0)

    return X_train, y_train, X_test, y_test

In [228]:
def main():
    all_data = preprocess_all_by_file(DATA_FILES, base_path="road/")
    train_files, test_files = split_files_by_session(all_data, test_ratio=0.20)
    print("Train sessions:", train_files)
    print("Test sessions:", test_files)
    # 3. Build final arrays
    X_train, y_train, X_test, y_test = build_dataset_from_split(
        all_data, train_files, test_files
    )
    print("X_train:", X_train.shape)
    print("X_test:", X_test.shape)
    #save the dataset 
    np.savez_compressed(
        "roads_canids_windows_200hz_3s.npz",
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test
    )   
    

In [229]:
if __name__ == '__main__':
    main()

Reading in:  road\road\signal_extractions\ambient\accelerator_attack_drive_1.csv
Reading in:  road\road\signal_extractions\ambient\accelerator_attack_drive_2.csv
Reading in:  road\road\signal_extractions\ambient\accelerator_attack_reverse_1.csv
Reading in:  road\road\signal_extractions\ambient\accelerator_attack_reverse_2.csv
Reading in:  road\road\signal_extractions\ambient\ambient_dyno_drive_basic_long.csv
Reading in:  road\road\signal_extractions\ambient\ambient_dyno_drive_basic_short.csv
Reading in:  road\road\signal_extractions\ambient\ambient_dyno_drive_benign_anomaly.csv
Reading in:  road\road\signal_extractions\ambient\ambient_dyno_drive_extended_long.csv
Reading in:  road\road\signal_extractions\ambient\ambient_dyno_drive_extended_short.csv
Reading in:  road\road\signal_extractions\ambient\ambient_dyno_drive_radio_infotainment.csv
Reading in:  road\road\signal_extractions\ambient\ambient_dyno_drive_winter.csv
Reading in:  road\road\signal_extractions\ambient\ambient_dyno_exerc

In [230]:
data = np.load("roads_canids_windows_200hz_3s.npz")

# Access arrays by their keys
X_train = data["X_train"]
y_train = data["y_train"]
X_test  = data["X_test"]
y_test  = data["y_test"]

# Optional: close the file handle
data.close()


In [233]:
X_train.columns

AttributeError: 'numpy.ndarray' object has no attribute 'columns'

In [235]:
from sklearn.utils.class_weight import compute_class_weight
#classes = np.unique(y_train) 
classes = np.array([0,1])   # force both classes to be considered

# class_weight = {0:1.0, 1: len(y_train)/max(1, np.sum(y_train==1))}
class_weight = {c: w for c,w in zip(classes, cw)}
print(class_weight)

{np.int64(0): np.float64(1.0)}


In [214]:
import numpy as np
from collections import Counter

print("y_train shape:", y_train.shape)
print("Label counts:", Counter(y_train))
print("Unique labels:", np.unique(y_train))
from collections import Counter
print("Train label distribution:", Counter(y_train))
print("Test  label distribution:", Counter(y_test))

y_train shape: (22503,)
Label counts: Counter({np.int64(0): 22503})
Unique labels: [0]
Train label distribution: Counter({np.int64(0): 22503})
Test  label distribution: Counter({np.int64(0): 2725})


In [215]:
# Two common strategies for CAN signal ML:
# Wide fixed feature vector (all signals from all IDs → keep NaNs → fill with 0)
# Flatten raw CAN frames into one feature vector (ID + each signal)
# Optionally add one-hot of ID (recommended)


In [216]:
# For supervised 1D-CNN, the simplest + most effective is:
# Sort by Time → resample to uniform Δt → forward-fill missing signals → set remaining NaNs to zero → window.